# 📘 Notebook 05: Grammars and Parsing

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module B: The Structure of Language · Notebook 2 of 3 in this module · (5 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **constituent** is and why sentences have tree structure
- Define a **context-free grammar** and use one to **generate** sentences
- **Parse** a sentence with NLTK and read the resulting **parse tree**
- Recognise **structural ambiguity** and measure how fast it grows
- Implement the **CKY algorithm** by hand
- Explain how a **probabilistic** grammar chooses between competing parses, and where its probabilities come from

> **Prerequisites:** Notebook 04. Recursion (a function that calls itself).
>
> 🔭 **Why this notebook matters:** *"the dog bit the man"* and *"the man bit the dog"* contain the same words. What differs is who did what to whom, and that information lives in the **structure** of the sentence. This notebook introduces the classical tool for describing that structure. It also explains why the hand-written approach, for all its elegance, eventually had to make room for statistics.

> ℹ️ **Setup note.** Run this cell once. It downloads a small sample of parsed sentences used in Section 6.

In [ ]:
import nltk
nltk.download("treebank", quiet=True)

import random
from collections import Counter

## 1. Sentences are not flat

### Intuition first
Take the sentence *"the angry dog chased a cat"*. You can replace *"the angry dog"* with the single word *"it"*, and *"a cat"* with *"them"*, and the sentence still works: *"it chased them"*. You cannot do the same with *"dog chased a"*. Some groups of neighbouring words behave as a unit and others do not.

A group of words that behaves as a unit is called a **constituent**. Constituents nest inside one another like boxes: *"a cat"* is a unit, *"chased a cat"* is a larger unit that contains it, and the whole sentence contains both. The natural picture of boxes inside boxes is a **tree**.

```
                 S
         ________|________
        NP                VP
   _____|_____         ___|____
 Det   Adj    N       V        NP
  |     |     |       |      __|__
 the  angry  dog   chased   Det   N
                             |    |
                             a   cat
```

The labels are the traditional names of the units: **S** for sentence, **NP** for noun phrase, **VP** for verb phrase, **Det** for determiner, **Adj** for adjective, **N** for noun and **V** for verb.

## 2. Context-free grammars

### Intuition first
A grammar is a set of recipes. Each recipe says how one kind of unit can be built from smaller ones: *"a sentence is a noun phrase followed by a verb phrase"*, *"a noun phrase is a determiner followed by a noun"*. Follow the recipes downwards until only words are left, and you have built a sentence.

### Formal definition
A **context-free grammar (CFG)** consists of:

1. a set of **non-terminal** symbols, the names of units: S, NP, VP ...
2. a set of **terminal** symbols, the actual words
3. a set of **rules** (or *productions*) of the form $A \rightarrow \beta$, where $A$ is one non-terminal and $\beta$ is a sequence of terminals and non-terminals
4. a **start symbol**, usually S

It is called *context-free* because a rule for $A$ can be applied wherever $A$ appears, regardless of what surrounds it.

We can store a grammar in a dictionary: each non-terminal maps to the list of its possible expansions.

In [ ]:
grammar = {
    "S":   [["NP", "VP"]],
    "NP":  [["Det", "N"], ["Det", "Adj", "N"]],
    "VP":  [["V", "NP"], ["V"]],
    "Det": [["the"], ["a"]],
    "Adj": [["small"], ["angry"], ["clever"]],
    "N":   [["dog"], ["cat"], ["student"]],
    "V":   [["chased"], ["saw"], ["slept"]],
}

### Generating sentences
To generate a sentence, start from `S` and keep replacing each non-terminal with one of its expansions, chosen at random, until only words remain. The function below does this with recursion: to expand a symbol, expand each part of the chosen recipe.

In [ ]:
def generate(symbol):
    if symbol not in grammar:          # a word: nothing left to expand
        return [symbol]
    expansion = random.choice(grammar[symbol])
    words = []
    for part in expansion:
        words += generate(part)
    return words

random.seed(3)
for i in range(6):
    print(" ".join(generate("S")))

Seven small rules produce a variety of sentences, all with correct word order. Look closely, though: one of them says *"a angry student"*. The grammar does not know that *a* becomes *an* before a vowel. Delete the `random.seed(3)` line and run the cell a few more times, and sooner or later someone will have *"slept the cat"*, because the grammar does not know that *slept* takes no object either. It **over-generates**: it produces strings that are not English.

> ⚠️ **Common pitfalls**
>
> - Believing a grammar describes meaning. *"the clever cat chased a small student"* is perfectly grammatical. A grammar licenses **structure**, and nothing more.
> - Under-estimating the detail a real grammar needs. To stop *"slept the cat"* you must split verbs into those that take an object and those that do not. Then singular and plural must agree, then tenses, and so on. Every fix multiplies the rules.

### Finite rules, infinite language
One more rule changes the nature of the grammar. Let a noun phrase contain a prepositional phrase, which itself contains a noun phrase.

In [ ]:
grammar["NP"].append(["NP", "PP"])
grammar["PP"] = [["P", "NP"]]
grammar["P"] = [["in"], ["with"], ["near"]]
grammar["N"] += [["park"], ["telescope"]]

random.seed(11)
for i in range(6):
    print(" ".join(generate("S")))

`NP` is now defined in terms of itself: a rule like this is called **recursive**. A noun phrase can contain a noun phrase that contains a noun phrase, without limit, so this handful of rules describes an **infinite** set of sentences. That is the central insight of generative grammar: a finite device accounts for an unbounded language. It is also precisely the nesting that the finite automata of Notebook 04 could not handle.

## 3. Parsing: from a sentence back to its tree

### Intuition first
Generation runs the grammar forwards: from `S` down to words. **Parsing** runs it backwards: given the words, find the tree. It is like being handed a finished dish and working out which recipes were used.

NLTK lets us write a grammar as text and gives us ready-made parsers. Words go in quotes, and `|` separates alternatives.

In [ ]:
toy = nltk.CFG.fromstring("""
    S   -> NP VP
    NP  -> Det N | NP PP | 'I'
    VP  -> V NP | VP PP
    PP  -> P NP
    Det -> 'the' | 'a'
    N   -> 'man' | 'dog' | 'telescope' | 'park'
    V   -> 'saw' | 'walked'
    P   -> 'with' | 'in'
""")

parser = nltk.ChartParser(toy)

sentence = "the dog saw a man".split()
for tree in parser.parse(sentence):
    print(tree)
    tree.pretty_print()

The first output is the tree in **bracket notation**: each pair of brackets is a constituent, labelled with its type. The second is the same tree drawn out. Both say that *"the dog"* is a noun phrase, *"saw a man"* is a verb phrase, and together they make a sentence.

A parser also tells us when a string is **not** in the language.

In [ ]:
def count_parses(text):
    return len(list(parser.parse(text.split())))

print(count_parses("the dog saw a man"))
print(count_parses("dog the saw man a"))     # the same words, shuffled

The shuffled version has no parse: it is not a sentence of this grammar. This is the membership question of Notebook 04, now answered for a context-free language.

## 4. Ambiguity

### Intuition first
Now a famous sentence: *"I saw the man with the telescope"*. Who has the telescope? Perhaps I used it to see him. Perhaps he was carrying it. The words are the same in both readings. The structures are not.

In [ ]:
sentence = "I saw the man with the telescope".split()
trees = list(parser.parse(sentence))

print(len(trees), "parses\n")
for tree in trees:
    tree.pretty_print()

Compare the two trees. In one, the phrase *"with the telescope"* (the PP) hangs from the **VP**: it describes the seeing. In the other it hangs from the **NP** *"the man"*: it describes the man. The grammar allows both, and nothing in the grammar says which is intended.

### Formal definition
A sentence is **structurally ambiguous** if the grammar assigns it more than one parse tree. This particular kind is called **PP-attachment ambiguity**, and it is everywhere in real text.

How bad does it get? Let us keep adding prepositional phrases.

In [ ]:
text = "I saw the man"
for extra in ["with the telescope", "in the park", "with the dog", "in the park", "with a telescope"]:
    text = text + " " + extra
    print(f"{len(text.split()):>2} words  {count_parses(text):>4} parses   {text}")

The counts are 2, 5, 14, 42, 132. These are the **Catalan numbers**, and they grow exponentially. An ordinary newspaper sentence of 25 words can have thousands or millions of parses under a realistic grammar. A human reader notices one, occasionally two.

This is the first of two discoveries that reshaped the field. Ambiguity is not an occasional curiosity in language. It is the normal case, and people resolve it constantly without being aware of it, using meaning and likelihood. A grammar alone cannot.

## 5. Parsing by hand: the CKY algorithm

### Intuition first
How does a parser find the trees without trying every possible one? The classical answer builds the solution from the bottom up. First work out what each **single word** can be. Then what each pair of neighbouring words can be, by combining the single-word answers. Then each stretch of three words, and so on, always reusing the answers for the shorter stretches. Results are stored in a table so that nothing is computed twice. This technique is called **dynamic programming**, and the algorithm is named **CKY** after Cocke, Kasami and Younger.

### Formal definition
CKY requires a grammar in **Chomsky normal form**, where every rule is either $A \rightarrow B\,C$ (exactly two non-terminals) or $A \rightarrow \text{word}$. Our toy grammar already has that shape. For a sentence of $n$ words the table has a cell for every span $(i, j)$, holding the set of non-terminals that can produce words $i$ to $j$:

- **Spans of one word:** look the word up.
- **Longer spans:** for every way to cut the span into a left part and a right part, and every rule $A \rightarrow B\,C$ with $B$ in the left cell and $C$ in the right cell, add $A$.

The sentence is grammatical if `S` is in the cell covering the whole sentence.

In [ ]:
binary_rules = {
    ("NP", "VP"): ["S"],
    ("Det", "N"): ["NP"],
    ("NP", "PP"): ["NP"],
    ("V", "NP"): ["VP"],
    ("VP", "PP"): ["VP"],
    ("P", "NP"): ["PP"],
}
lexicon = {
    "I": ["NP"], "the": ["Det"], "a": ["Det"],
    "man": ["N"], "dog": ["N"], "telescope": ["N"], "park": ["N"],
    "saw": ["V"], "walked": ["V"], "with": ["P"], "in": ["P"],
}

def cky(words):
    n = len(words)
    table = {}
    for i in range(n):                                   # spans of one word
        table[(i, i + 1)] = set(lexicon.get(words[i], []))

    for length in range(2, n + 1):                       # longer and longer spans
        for start in range(0, n - length + 1):
            end = start + length
            cell = set()
            for split in range(start + 1, end):          # every way to cut the span in two
                for left in table[(start, split)]:
                    for right in table[(split, end)]:
                        cell.update(binary_rules.get((left, right), []))
            table[(start, end)] = cell
    return table

words = "I saw the man with the telescope".split()
table = cky(words)

for (start, end), cell in sorted(table.items(), key=lambda item: (item[0][1] - item[0][0], item[0][0])):
    if cell:
        print(f"{' '.join(words[start:end]):<34} {sorted(cell)}")

Read the output from top to bottom: single words first, then longer spans. Spans that are not constituents, such as *"saw the"*, have empty cells and are not shown. The last line covers the whole sentence and contains `S`.

In [ ]:
def is_sentence(text):
    words = text.split()
    return "S" in cky(words)[(0, len(words))]

for text in ["the dog saw a man", "I walked the dog in the park", "dog the saw man a", "I saw"]:
    print(f"{text:<36} {is_sentence(text)}")

The last one, *"I saw"*, is rejected because our small grammar insists that every verb has an object.

The three nested loops over `length`, `start` and `split` mean the work grows with the **cube** of the sentence length. That is far better than exponential: CKY decides whether a sentence is grammatical in polynomial time even when the number of parse trees is astronomical, because the table shares the common pieces between them.

## 6. Choosing a parse: probabilistic grammars

### Intuition first
The parser found two trees for the telescope sentence. To pick one, we need to know which structures are more **likely**. The simplest idea is to attach a probability to every rule. The probability of a tree is then the product of the probabilities of the rules used to build it, and the best parse is the tree with the highest probability.

### Formal definition
A **probabilistic context-free grammar (PCFG)** is a CFG in which every rule has a probability, and the probabilities of all rules with the same left-hand side add up to 1.

In [ ]:
pcfg = nltk.PCFG.fromstring("""
    S   -> NP VP        [1.0]
    NP  -> Det N        [0.5] | NP PP [0.2] | 'I' [0.3]
    VP  -> V NP         [0.7] | VP PP [0.3]
    PP  -> P NP         [1.0]
    Det -> 'the'        [0.7] | 'a' [0.3]
    N   -> 'man'        [0.4] | 'dog' [0.2] | 'telescope' [0.2] | 'park' [0.2]
    V   -> 'saw'        [0.6] | 'walked' [0.4]
    P   -> 'with'       [0.6] | 'in' [0.4]
""")

best_parser = nltk.ViterbiParser(pcfg)

for tree in best_parser.parse("I saw the man with the telescope".split()):
    print(f"Probability of the best tree: {tree.prob():.6f}")
    tree.pretty_print()

The parser returns a single tree, the most probable one. Here it attaches *"with the telescope"* to the verb. The reason is visible in the grammar: the two trees use exactly the same rules except for one. The verb reading uses `VP -> VP PP`, with probability 0.3, and the noun reading uses `NP -> NP PP`, with probability 0.2.

### Where do the probabilities come from?
We invented those numbers. In practice they are **counted**. Linguists have annotated large collections of sentences with their correct trees. Such a collection is called a **treebank**. NLTK includes a small sample of the best known one, the Penn Treebank, made from Wall Street Journal articles.

In [ ]:
from nltk.corpus import treebank

print(len(treebank.parsed_sents()), "parsed sentences\n")
print(" ".join(treebank.sents()[0]), "\n")
treebank.parsed_sents()[0].pretty_print()

A real tree is considerably more detailed than our toy ones: more categories, and finer distinctions. To turn a treebank into a PCFG, count how often each rule is used and divide by the number of times its left-hand side appears.

In [ ]:
rule_counts = Counter()
for tree in treebank.parsed_sents():
    for production in tree.productions():
        if production.is_nonlexical():           # skip the rules that introduce words
            rule_counts[production] += 1

np_total = 0
for production, count in rule_counts.items():
    if str(production.lhs()) == "NP":
        np_total += count

print(f"{len(rule_counts):,} different rules in {len(treebank.parsed_sents()):,} sentences\n")
print("The most common ways to build an NP:")
shown = 0
for production, count in rule_counts.most_common():
    if str(production.lhs()) == "NP":
        print(f"  {str(production):<26} {count:>5}   probability {count / np_total:.3f}")
        shown += 1
        if shown == 8:
            break

Thousands of distinct rules, from only a few thousand sentences. This is the second discovery. Nobody could write that grammar by hand, and nobody has to: **the grammar and its probabilities can be read off annotated data**.

> 🧠 This is the turning point of the whole field, and of this course. For decades NLP meant writing rules by hand. From the 1990s it increasingly meant **counting** and, later, **learning**. The remaining notebooks follow that path. We will not write another grammar. We will let data tell us what is likely.

---
## ✏️ Exercises

### Exercise 1 (Grow the generator)
Extend the dictionary `grammar` from Section 2 so that a verb phrase may end with an **adverb** (`quickly`, `quietly`, `happily`). Generate eight sentences.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
grammar["VP"].append(["VP", "Adv"])
grammar["Adv"] = [["quickly"], ["quietly"], ["happily"]]

random.seed(5)
for i in range(8):
    print(" ".join(generate("S")))
```

*The new rule is recursive, so a sentence may end in several adverbs in a row. One line added to the grammar, an unlimited number of new sentences.*
</details>

### Exercise 2 (A grammar for a familiar language)
In Notebook 04 we showed that no finite automaton can recognise $a^n b^n$. Write a context-free grammar for it with `nltk.CFG.fromstring`, using a single non-terminal. Check that it parses `a a b b` and `a a a b b b`, and that it does not parse `a a b`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
anbn = nltk.CFG.fromstring("""
    S -> 'a' S 'b' | 'a' 'b'
""")
anbn_parser = nltk.ChartParser(anbn)

for text in ["a a b b", "a a a b b b", "a a b"]:
    print(f"{text:<12} {len(list(anbn_parser.parse(text.split())))} parse(s)")
```

*Each use of the rule adds one a on the left and one b on the right at the same time, so the two counts can never differ. One recursive rule does what no number of states could.*
</details>

### Exercise 3 (An elephant in my pyjamas)
Groucho Marx said: *"I shot an elephant in my pyjamas. How he got in my pyjamas, I don't know."* Write a small NLTK grammar that can parse `I shot an elephant in my pyjamas`, and show that the sentence has two parses. Which tree corresponds to the joke?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
groucho = nltk.CFG.fromstring("""
    S   -> NP VP
    NP  -> Det N | NP PP | 'I'
    VP  -> V NP | VP PP
    PP  -> P NP
    Det -> 'an' | 'my'
    N   -> 'elephant' | 'pyjamas'
    V   -> 'shot'
    P   -> 'in'
""")

trees = list(nltk.ChartParser(groucho).parse("I shot an elephant in my pyjamas".split()))
print(len(trees), "parses")
for tree in trees:
    tree.pretty_print()
```

*The sensible reading attaches "in my pyjamas" to the verb phrase: I was in my pyjamas when I shot. The joke is the other tree, where the phrase attaches to "an elephant". The grammar finds both equally acceptable. Knowing that elephants do not wear pyjamas is not grammar. It is knowledge of the world.*
</details>

### Exercise 4 (Flip the decision)
In the PCFG of Section 6, change the probabilities so that the parser prefers to attach *"with the telescope"* to **the man**. Remember that the probabilities of rules with the same left-hand side must still add up to 1.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
pcfg_2 = nltk.PCFG.fromstring("""
    S   -> NP VP        [1.0]
    NP  -> Det N        [0.4] | NP PP [0.4] | 'I' [0.2]
    VP  -> V NP         [0.9] | VP PP [0.1]
    PP  -> P NP         [1.0]
    Det -> 'the'        [0.7] | 'a' [0.3]
    N   -> 'man'        [0.4] | 'dog' [0.2] | 'telescope' [0.2] | 'park' [0.2]
    V   -> 'saw'        [0.6] | 'walked' [0.4]
    P   -> 'with'       [0.6] | 'in' [0.4]
""")

for tree in nltk.ViterbiParser(pcfg_2).parse("I saw the man with the telescope".split()):
    tree.pretty_print()
```

*With `NP -> NP PP` now more probable than `VP -> VP PP`, the phrase attaches to the noun. Notice the weakness: this grammar will now attach every prepositional phrase to a noun, including in "I saw the man with my own eyes". A plain PCFG cannot look at which words are involved. Better models can.*
</details>

### Exercise 5 (Count the parses with CKY, a little harder)
Change `cky` into `cky_count(words)`, which stores in each cell a dictionary from non-terminal to the **number of ways** it can be built. Return the number of parses of the whole sentence, and check it against NLTK for the sentence with three prepositional phrases. (Hint: the number of ways to build $A$ from a left part $B$ and a right part $C$ is the product of their counts, and you add up over all cuts and rules.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def cky_count(words):
    n = len(words)
    table = {}
    for i in range(n):
        table[(i, i + 1)] = Counter(lexicon.get(words[i], []))

    for length in range(2, n + 1):
        for start in range(0, n - length + 1):
            end = start + length
            cell = Counter()
            for split in range(start + 1, end):
                for left, left_ways in table[(start, split)].items():
                    for right, right_ways in table[(split, end)].items():
                        for parent in binary_rules.get((left, right), []):
                            cell[parent] += left_ways * right_ways
            table[(start, end)] = cell
    return table[(0, n)]["S"]

text = "I saw the man with the telescope in the park with the dog"
print("CKY: ", cky_count(text.split()))
print("NLTK:", count_parses(text))
```

*The table counts 14 trees without ever building one of them. The same trick, with "multiply and take the maximum" in place of "multiply and add", finds the most probable parse of a PCFG. That is what `ViterbiParser` does, and you will meet the idea again in the next notebook.*
</details>

### Exercise 6 (Read a real grammar)
Using `rule_counts` from Section 6, print the eight most common rules whose left-hand side is `VP`, with their probabilities. What does the list tell you about how English verb phrases are built?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
vp_total = 0
for production, count in rule_counts.items():
    if str(production.lhs()) == "VP":
        vp_total += count

shown = 0
for production, count in rule_counts.most_common():
    if str(production.lhs()) == "VP":
        print(f"{str(production):<28} {count:>5}   {count / vp_total:.3f}")
        shown += 1
        if shown == 8:
            break
```

*The tags starting with VB are verb forms (VBD is the past tense, VBZ the third person singular, VB the base form). TO is the word "to" and MD is a modal such as "will" or "can". The commonest patterns are "to" or a modal followed by another verb phrase, a verb followed by a noun phrase, and a verb followed by a clause (SBAR). Even the most frequent rule has a probability below 0.1: the rest is spread over a great many rare rules, which is another face of Zipf's law.*
</details>

## 🔑 Key takeaways

- Sentences have **tree structure**: words group into **constituents**, which nest inside larger ones.
- A **context-free grammar** is a set of rules that rewrite one symbol into a sequence. **Recursive** rules let a finite grammar describe an infinite language.
- **Parsing** recovers the tree of a sentence. **CKY** does it bottom-up with a table, in time proportional to the cube of the sentence length.
- **Ambiguity is the normal case.** The number of parses can grow exponentially with sentence length.
- A **PCFG** attaches probabilities to rules and selects the most probable tree.
- Grammars and probabilities can be **counted from a treebank**. Writing them by hand does not scale.
- A grammar captures structure. It does not capture meaning or knowledge of the world.

---
**Next:** *Notebook 06: Part-of-Speech Tagging and Named Entities*, where we take the lesson of this notebook to heart and build our first system that learns entirely from annotated data.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*